# OpenOppsDB — Advanced usage

Durable joins, version history, a company drill-down, and when to use Parquet exports. SQL magics run through DuckDB attached to the read-only SQLite snapshot.

## Contents

- Setup (read-only `/kaggle/input`, `mode=ro&immutable=1`)
- Queries and charts for this kernel
- Links to the rest of the collection

## Collection

| Notebook | Kernel | What it is for |
| --- | --- | --- |
| **Starter** | [`wyattowalsh/openoppsdb-starter-notebook`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-starter-notebook) | Front door: tables, recent open jobs, first `%%sql` cells |
| **Explorer (featured)** | [`wyattowalsh/openoppsdb-explorer`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-explorer) | Gradio UI: jobs, companies, skills, filters/plots |
| **Advanced usage** | [`wyattowalsh/openoppsdb-advanced-usage`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-advanced-usage) | Joins, version history, company drill-down, Parquet |
| **SQL playground** | [`wyattowalsh/openoppsdb-sql-playground`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-sql-playground) | JupySQL studio: CTEs, DuckDB attach, Parquet scans |
| **Hiring market map** | [`wyattowalsh/openoppsdb-hiring-market-map`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-hiring-market-map) | Company, provider, location, and remote mix charts |
| **Skills radar** | [`wyattowalsh/openoppsdb-skills-radar`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-skills-radar) | Skill groups, keywords, and co-occurrence |
| **Snapshot health** | [`wyattowalsh/openoppsdb-snapshot-health`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-snapshot-health) | Coverage, freshness, sync runs, observation mix |


In [ ]:
%pip install -q jupysql==0.11.1 duckdb==1.5.5 duckdb-engine==0.17.0 plotly==7.0.0


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
ROUTE_LEDGER = {
    "pine": "#2f6f50",
    "paper": "#f7f1df",
    "brass": "#d99629",
    "ink": "#1d281f",
    "info": "#336d8f",
}

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
PARQUET_DIR = DATASET_DIR / "exports" / "parquet"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 25
%config SqlMagic.autolimit = 100


In [ ]:
import duckdb

con = duckdb.connect()
attach_path = str(DB_PATH).replace("'", "''")
con.execute(f"ATTACH '{attach_path}' AS oo (TYPE SQLITE, READ_ONLY)")
print("DuckDB attached the read-only SQLite snapshot as oo")


In [ ]:
%sql con --alias openopps


In [ ]:
%%sql
SELECT j.id AS job_id,
       coalesce(v.company, b.name) AS company,
       v.title, j.provider_id, j.status, v.remote,
       v.employment_type, j.first_seen_at, j.last_seen_at,
       v.posting_url
FROM oo.jobs j
JOIN oo.job_versions v ON v.id = j.current_version_id
LEFT JOIN oo.boards b ON b.key = j.board_key
WHERE j.status = 'open'
ORDER BY j.last_seen_at DESC
LIMIT 25


In [ ]:
%%sql --save version_history
SELECT j.id AS job_id,
       coalesce(max(v.company), max(b.name)) AS company,
       max(v.title) AS latest_title,
       count(v.id) AS version_count,
       min(v.first_seen_at) AS first_version_seen_at,
       max(v.last_seen_at) AS last_version_seen_at
FROM oo.jobs j
JOIN oo.job_versions v ON v.job_id = j.id
LEFT JOIN oo.boards b ON b.key = j.board_key
GROUP BY j.id
HAVING count(v.id) > 1
ORDER BY version_count DESC, last_version_seen_at DESC
LIMIT 20


In [ ]:
%%sql
SELECT observation_kind, count(*) AS observations
FROM oo.job_sync_observations
GROUP BY observation_kind
ORDER BY observations DESC


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    companies = pd.read_sql_query(
        """
        select coalesce(v.company, b.name, 'Unknown') as company,
               count(*) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        left join boards b on b.key = j.board_key
        where j.status = 'open'
        group by company
        order by open_roles desc, company
        limit 15
        """,
        conn,
    )
focus_company = companies["company"].iloc[0] if not companies.empty else None
if focus_company:
    with sqlite3.connect(DB_URI, uri=True) as conn:
        dossier = pd.read_sql_query(
            """
            select j.id as job_id, v.title, j.provider_id, v.remote,
                   v.employment_type, j.last_seen_at, v.posting_url
            from jobs j
            join job_versions v on v.id = j.current_version_id
            left join boards b on b.key = j.board_key
            where j.status = 'open'
              and coalesce(v.company, b.name, 'Unknown') = ?
            order by j.last_seen_at desc
            limit 25
            """,
            conn,
            params=(focus_company,),
        )
    print(f"Company drill-down: {focus_company}")
    display(companies)
    display(dossier)
else:
    print("No open-role companies in this snapshot.")
    companies


In [ ]:
parquet_path = PARQUET_DIR / "job_versions.parquet"
if parquet_path.exists():
    sample = con.execute(
        """
        select id, title, company, left(cast(description as varchar), 240) as description_head,
               posting_url
        from read_parquet(?)
        where description is not null
        limit 10
        """,
        [str(parquet_path)],
    ).df()
else:
    sample = pd.DataFrame({"note": ["Parquet export not found"]})
sample
